In [1]:
from torch_harmonics.spherical_harmonics import SphericalHarmonics

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from torch_harmonics import plotting
from torch.utils.data import DataLoader
import xarray
import torch
from torch import nn

In [3]:
f = xarray.load_dataset('/home/colin/hdd/workspace/datasets/test_dataset.nc')

In [4]:
dataset = f['rti']
dataset = torch.tensor(dataset.to_numpy())[:,:,:,:,0].view(-1,61,21,107).repeat(1,2,1,1)
dataset = dataset.permute(0,3,1,2)

In [5]:
B = 1
dataloader = DataLoader(dataset, batch_size=B, shuffle=True)

In [6]:
next(iter(dataloader)).shape

torch.Size([1, 107, 122, 21])

In [7]:
class SHT(nn.Module):
    def __init__(self, L=3, num_theta=61*2, num_phi=21, num_range=107):
        super().__init__()
        self.num_range = num_range
        self.w_lin = nn.Linear(num_range*num_theta*num_phi, num_range*(L+1)**2)
        self.sh = SphericalHarmonics(L, num_lat=num_theta, num_lon=num_phi)
        #self.mlp_out = nn.Linear(num_theta*num_phi, num_theta*num_phi)

    def forward(self, x):
        B = x.size(0)
        
        w = self.w_lin(x)
        w = w.view(-1, (L+1)**2)
        y = self.sh(w).float().view(B, self.num_range, self.sh.num_lat, self.sh.num_lon)
        #y = self.mlp_out(y).view(1,self.sh.num_lat, self.sh.num_lon)
        return y, w.detach()

In [8]:
L = 3
sh_model = SHT(L)
optimizer = torch.optim.Adam(sh_model.parameters(), lr = 1e-4)
num_epochs = 10

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in dataloader:
        y_pred, _ = sh_model(batch.view(B, -1))
        loss = (y_pred - batch).pow(2).mean()
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    print(f"Loss: {epoch_loss / len(dataloader)}")

Loss: 46.64143665000649
Loss: 45.58799746984228
Loss: 45.51357295171637


KeyboardInterrupt: 

In [9]:
plotting.plot_spherical_fn(pred.squeeze().numpy(), fig=plt.figure(figsize=(2,2)))

torch.Size([1, 107, 122, 21])